In this notebook, we evaluate the trained models stored in `./models/` on the
test data.
We begin by loading the data and defining a helper function to compute and
print the official RMS errors for submission to
[nonlinearbenchmarks.org](https://www.nonlinearbenchmark.org/).

In [ ]:
import freq_statespace as fss
import nonlinear_benchmarks as nlb  # pip install git+https://github.com/MaartenSchoukens/nonlinear_benchmarks.git
from nonlinear_benchmarks.error_metrics import RMSE, NRMSE
import numpy as np

data_train, data_test = nlb.FineSteeringMirror()
n_init = data_test[0].state_initialization_window_length

def print_offical_benchmark_results(y_test_measured, y_test_simulated, n_init):
    """Prints the official RMSEs on the test sets.
    
    According to https://github.com/MaartenSchoukens/nonlinear_benchmarks/blob/master/submission_examples/FineSteeringMirror.py.
    """
    all_RMSEs = []
    all_NRMSEs = []
    for _, test, simulation in zip(range(len(y_test_measured)), y_test_measured, y_test_simulated):
        test_RMSE = 1e6 * RMSE(test.y, simulation, n_init=n_init)
        test_NRMSE = NRMSE(test.y, simulation, n_init=n_init)
        
        # Average over all periods and realizations
        test_RMSE_mean = np.mean(test_RMSE, axis=(1, 2))
        test_NRMSE_mean = np.mean(test_NRMSE, axis=(1, 2))
        
        all_RMSEs.append(test_RMSE_mean)
        all_NRMSEs.append(test_NRMSE_mean)
            
    for test, test_RMSE, test_NRMSE in zip(y_test_measured, all_RMSEs, all_NRMSEs):
        print(f"  {test.name}:")
        print(f"    RMSE to submit: {np.mean(test_RMSE):.3e} µm ({100*np.mean(test_NRMSE):.2f}% relative error)")

First, we assess how BLA models trained on individual amplitude levels generalize to all test amplitude levels.

In [2]:
amplitude_list = [100, 200, 300]
for amplitude in amplitude_list:
    bla = fss.load_model(f"models/bla_{amplitude}mV.zip")
    y_test_simulated = []
    
    print(f"BLA trained on {amplitude} mV data:")
    for test in data_test:
        # Below, `offset` suppresses the initial transient in the simulated output.
        # It effectively acts as a simple (somewhat hacky) form of initial state
        # estimation, which works in this setting thanks to the periodic nature of
        # the data (see the `model.simulate()` docstring for details).
        y_test_simulated.append(bla.simulate(test.u, offset=1000)[0])
    print_offical_benchmark_results(data_test, y_test_simulated, n_init=n_init)
    print()

BLA trained on 100 mV data:
  test 100mV:
    RMSE to submit: 1.142e-01 µm (8.38% relative error)
  test 200mV:
    RMSE to submit: 5.548e-01 µm (20.50% relative error)
  test 300mV:
    RMSE to submit: 1.319e+00 µm (32.71% relative error)

BLA trained on 200 mV data:
  test 100mV:
    RMSE to submit: 2.963e-01 µm (21.78% relative error)
  test 200mV:
    RMSE to submit: 2.161e-01 µm (7.95% relative error)
  test 300mV:
    RMSE to submit: 6.987e-01 µm (17.34% relative error)

BLA trained on 300 mV data:
  test 100mV:
    RMSE to submit: 4.459e-01 µm (32.84% relative error)
  test 200mV:
    RMSE to submit: 4.119e-01 µm (15.17% relative error)
  test 300mV:
    RMSE to submit: 2.277e-01 µm (5.63% relative error)



The above results are expected for a weakly nonlinear system. Each
individual BLA performs well at the amplitude level it was trained on, but
fails to generalize to other test sets due to amplitude-dependent resonance
behavior. Note that fitting a single linear model to the combined training data does
not resolve this issue, as shown below.

In [3]:
bla = fss.load_model(f"models/bla_all_amplitudes.zip")
y_test_simulated = []

print(f"Single BLA trained on all amplitudes:")
for test in data_test:
    y_test_simulated.append(bla.simulate(test.u, offset=1000)[0])
print_offical_benchmark_results(data_test, y_test_simulated, n_init=n_init)

Single BLA trained on all amplitudes:
  test 100mV:
    RMSE to submit: 2.523e-01 µm (18.61% relative error)
  test 200mV:
    RMSE to submit: 1.790e-01 µm (6.61% relative error)
  test 300mV:
    RMSE to submit: 6.575e-01 µm (16.28% relative error)


It is not surprising that the combined BLA performs best at the central
(middle) amplitude level, as this level most closely matches the averaged
nonparametric BLA obtained across all amplitudes. To obtain a single model
that performs well across all amplitude levels, a nonlinear approach is
required:

In [4]:
nllfr = fss.load_model(f"models/nllfr_all_amplitudes.zip")
y_test_simulated = []

print(f"Single NL-LFR trained on all amplitudes:")
for test in data_test:
    y_test_simulated.append(nllfr.simulate(test.u, offset=1000)[0])
print_offical_benchmark_results(data_test, y_test_simulated, n_init=n_init)

Single NL-LFR trained on all amplitudes:
  test 100mV:
    RMSE to submit: 8.662e-02 µm (6.37% relative error)
  test 200mV:
    RMSE to submit: 1.309e-01 µm (4.84% relative error)
  test 300mV:
    RMSE to submit: 1.686e-01 µm (4.19% relative error)
